**Notebook roadmap (below):** load Moodify Excel → profile CIDs → call the same PubChem + parser stack as `bp_vp_construct.ipynb` → merge BP/VP onto every ingredient row → save CSV → print success-rate summary → **Part D:** merge `data/raw_dragon_matrix.csv`, build the same **54** standardized Dragon PCs as `features_set_1.ipynb` / `classifications_trials.ipynb` → rank by |Pearson r| vs boiling point → `X_top_df` / `X_top_arr` → Ridge regression and error metrics.

---

The idea is the follwing:
I do not have the full covering of "Waka" data set with experimental boiling point, 192 from 312. In ref.2 there is a claim that boiling point should be a good estimator for perceptual intncity. My idea is to use take single molecules list from the Moodify inventory ~2400 single molecules ingredients. Try to retrieve from pubchem Standard boiling point. Test the coverage, use Dragon-derived PC features (same pipeline as intensity classification), fit a regressor for boiling point, and inspect accuracy.

path to molecules data set with CID identifier project_1\data\Cannonical_names_for_moodify_db from pubchem.xlsx

In [7]:
# Libraries: pandas loads tables; pathlib builds paths that work on Windows/macOS/Linux.
from pathlib import Path

import pandas as pd

# Same pipeline as bp_vp_construct.ipynb:
#   batch_physical_props.fetch_props_for_dataframe
#     → pubchem_retriever2.get_pubchem_physical_props (PUG-View REST API)
#     → practical_parser2.parse_boiling_point (prefers normal BP near 760 mmHg → °C)
#     → practical_parser2.parse_vapor_pressure (prefers measurement near 25 °C → mmHg)
from batch_physical_props import fetch_props_for_dataframe, merge_props

DATA_DIR = Path("data")

In [8]:
# Reading .xlsx needs the `openpyxl` package in this kernel:  pip install openpyxl
xlsx_path = DATA_DIR / "Cannonical_names_for_moodify_db from pubchem.xlsx"
molecules_df = pd.read_excel(xlsx_path)
molecules_df.head()

,moodifyindex(mi),Name of Material,CAS,CID,SMILES,ConnectivitySMILES,InChI,InChIKey,IUPACName,retrieved_by
0,1000001,BENZALDEHYDE,100-52-7,240,C1=CC=C(C=C1)C=O,C1=CC=C(C=C1)C=O,InChI=1S/C7H6O/c8-6-7-4-2-1-3-5-7/h1-6H,HUMNYLRZRPPJDN-UHFFFAOYSA-N,benzaldehyde,CID
1,1000002,BOURGEONAL,18127-01-0,64832,CC(C)(C)C1=CC=C(C=C1)CCC=O,CC(C)(C)C1=CC=C(C=C1)CCC=O,"InChI=1S/C13H18O/c1-13(2,3)12-8-6-11(7-9-12)5-...",FZJUFJKVIYFBSY-UHFFFAOYSA-N,3-(4-tert-butylphenyl)propanal,CID
2,1000003,CAMPHOR,76-22-2,2537,CC1(C2CCC1(C(=O)C2)C)C,CC1(C2CCC1(C(=O)C2)C)C,"InChI=1S/C10H16O/c1-9(2)7-4-5-10(9,3)8(11)6-7/...",DSSYKIVIOFKYAU-UHFFFAOYSA-N,"1,7,7-trimethylbicyclo[2.2.1]heptan-2-one",CID
3,1000004,CEDROL METHYL ETHER,19870-74-7,88288,C[C@@H]1CC[C@@H]2[C@]13CC[C@]([C@H](C3)C2(C)C)...,CC1CCC2C13CCC(C(C3)C2(C)C)(C)OC,"InChI=1S/C16H28O/c1-11-6-7-12-14(2,3)13-10-16(...",HRGPYCVTDOECMG-WALBABNVSA-N,"(1S,2R,5S,7R,8S)-8-methoxy-2,6,6,8-tetramethyl...",CID
4,1000005,(E)-CITRAL,141-27-5,638011,CC(=CCC/C(=C/C=O)/C)C,CC(=CCCC(=CC=O)C)C,"InChI=1S/C10H16O/c1-9(2)5-4-6-10(3)7-8-11/h5,7...",WTEVQBCEXWBHNA-JXMROGBWSA-N,"(2E)-3,7-dimethylocta-2,6-dienal",CID


In [11]:
# Quick inventory: ingredient rows vs distinct PubChem CIDs.
# PubChem requests are one per *unique* CID; duplicate CIDs in this sheet are not re-fetched.
print("shape (rows, cols):", molecules_df.shape)
print("unique CID:", molecules_df["CID"].nunique())
print("rows with missing CID:", int(molecules_df["CID"].isna().sum()))

dup = molecules_df["CID"].value_counts()
print("CIDs used on more than one row:", int((dup > 1).sum()))

shape (rows, cols): (2326, 10)
unique CID: 2006
rows with missing CID: 0
CIDs used on more than one row: 242


### Step A — What PubChem gives us (same idea as `bp_vp_construct.ipynb`)

1. **Identifier** — We use **CID** (PubChem Compound ID). Your Moodify export already has it.

2. **Where the numbers come from** — `get_pubchem_physical_props` calls PubChem **PUG-View** for two section headings: *Boiling Point* and *Vapor Pressure*. The API returns free-text lines (often with units, pressures, temperatures).

3. **How we normalize ("standard-ish")** — `practical_parser2` turns that text into scalars:
   - **Boiling point** → degrees Celsius, preferring a value associated with **760 mmHg** (normal boiling point) when the text allows.
   - **Vapor pressure** → **mmHg**, preferring a measurement taken closest to **25 °C** (you can change `preferred_vp_temperature_c` if you need another reference temperature).

4. **Politeness & speed** — `delay_s` spaces requests so we do not hammer PubChem. **`resume=True`** appends each newly fetched CID to a **CSV cache**; if the run stops, the next run skips CIDs already stored.

5. **Shared cache** — We use the **same** `pubchem_physical_props_cache.csv` as the Waka notebook. Any CID already fetched there is reused **without** a new HTTP request. The `props_df` object you get back can therefore contain **more CIDs than Moodify** (it is the whole cache merged with this run); the summary cell later filters back to Moodify CIDs for compound-level percentages.

In [13]:
# Optional: show INFO lines from batch_physical_props (one line per newly fetched CID).
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# This returns ONE ROW PER UNIQUE CID (not per ingredient row).
# First run for ~2300 new CIDs can take on the order of tens of minutes (delay + network).
props_df = fetch_props_for_dataframe(
    molecules_df,
    cid_column="CID",
    delay_s=0.35,
    max_retries=4,
    retry_backoff_s=2.0,
    preferred_vp_temperature_c=25.0,
    cache_path=DATA_DIR / "pubchem_physical_props_cache.csv",
    resume=True,
)

props_df.head()

INFO: Fetched CID 64832 (1 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 2537 (2 of 2006 unique CIDs this run): bp=203.889 vp=0.65 err=-
INFO: Fetched CID 88288 (3 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 86209 (4 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 641256 (5 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 443158 (6 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 5283335 (7 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 998 (8 of 2006 unique CIDs this run): bp=None vp=0.39 err=-
INFO: Fetched CID 637563 (9 of 2006 unique CIDs this run): bp=234.5 vp=0.05 err=-
INFO: Fetched CID 14286 (10 of 2006 unique CIDs this run): bp=191.0 vp=0.37 err=-
INFO: Fetched CID 122510 (11 of 2006 unique CIDs this run): bp=None vp=None err=-
INFO: Fetched CID 26331 (12 of 2006 unique CIDs this run): bp=152.5 vp=1.67 err=-
INFO: Fetched CID 5371002 (1

,cid,boiling_point_c,vapor_pressure_mmhg,error
0,107,280.000,NaN,
1,125,252.000,0.0008,
2,126,310.500,NaN,
3,176,117.778,15.7000,
4,177,21.111,902.0000,


### Step B — Merge properties back onto every ingredient row

`props_df` is compact (unique CIDs). The Moodify table has **one row per ingredient**; several rows can share one CID.

`merge_props` **left-joins** the parsed columns onto `molecules_df` on `CID`, so every ingredient keeps its own row and gets the BP/VP for its compound. New columns:
- `bp_c_pubchem` — boiling point (°C)
- `vp_mmhg_pubchem` — vapor pressure (mmHg) near 25 °C when the parser could extract it
- `bp_c_pubchem_fetch_error` — non-empty if the HTTP/retry layer failed (not the same as "PubChem has no BP text")

In [14]:
molecules_with_props = merge_props(
    molecules_df,
    props_df,
    cid_column="CID",
    bp_column="bp_c_pubchem",
    vp_column="vp_mmhg_pubchem",
)

out_path = DATA_DIR / "moodify_inventory_pubchem_bp_vp.csv"
molecules_with_props.to_csv(out_path, index=False)
print("Wrote:", out_path.resolve())
molecules_with_props.head()

Wrote: C:\src\ai-program-2026\project_1\data\moodify_inventory_pubchem_bp_vp.csv


,moodifyindex(mi),Name of Material,CAS,CID,SMILES,ConnectivitySMILES,InChI,InChIKey,IUPACName,retrieved_by,bp_c_pubchem,vp_mmhg_pubchem,bp_c_pubchem_fetch_error
0,1000001,BENZALDEHYDE,100-52-7,240,C1=CC=C(C=C1)C=O,C1=CC=C(C=C1)C=O,InChI=1S/C7H6O/c8-6-7-4-2-1-3-5-7/h1-6H,HUMNYLRZRPPJDN-UHFFFAOYSA-N,benzaldehyde,CID,178.889,1.27,NaN
1,1000002,BOURGEONAL,18127-01-0,64832,CC(C)(C)C1=CC=C(C=C1)CCC=O,CC(C)(C)C1=CC=C(C=C1)CCC=O,"InChI=1S/C13H18O/c1-13(2,3)12-8-6-11(7-9-12)5-...",FZJUFJKVIYFBSY-UHFFFAOYSA-N,3-(4-tert-butylphenyl)propanal,CID,NaN,NaN,
2,1000003,CAMPHOR,76-22-2,2537,CC1(C2CCC1(C(=O)C2)C)C,CC1(C2CCC1(C(=O)C2)C)C,"InChI=1S/C10H16O/c1-9(2)7-4-5-10(9,3)8(11)6-7/...",DSSYKIVIOFKYAU-UHFFFAOYSA-N,"1,7,7-trimethylbicyclo[2.2.1]heptan-2-one",CID,203.889,0.65,
3,1000004,CEDROL METHYL ETHER,19870-74-7,88288,C[C@@H]1CC[C@@H]2[C@]13CC[C@]([C@H](C3)C2(C)C)...,CC1CCC2C13CCC(C(C3)C2(C)C)(C)OC,"InChI=1S/C16H28O/c1-11-6-7-12-14(2,3)13-10-16(...",HRGPYCVTDOECMG-WALBABNVSA-N,"(1S,2R,5S,7R,8S)-8-methoxy-2,6,6,8-tetramethyl...",CID,NaN,NaN,
4,1000005,(E)-CITRAL,141-27-5,638011,CC(=CCC/C(=C/C=O)/C)C,CC(=CCCC(=CC=O)C)C,"InChI=1S/C10H16O/c1-9(2)5-4-6-10(3)7-8-11/h5,7...",WTEVQBCEXWBHNA-JXMROGBWSA-N,"(2E)-3,7-dimethylocta-2,6-dienal",CID,228.500,1.00,NaN


In [ ]:
*(Optional)* Run **Step C** summary cell above first; **Part D** starts in the next cells.

### Step C — Success rates (how to read them)

- **Fetch layer** — `error` in `props_df` (shown as `bp_c_pubchem_fetch_error` on the merged table) means the script could not complete a parseable download after retries. Rare if the network behaves.

- **Coverage** — A **successful fetch** can still yield `NaN` for BP or VP if PubChem has no usable text for that field, or the parser could not extract a number.

We report: (1) **per unique CID** — what fraction of compounds have values; (2) **per ingredient row** — same but weighted by your table (duplicate CIDs count multiple times).

In [15]:
# `props_df` can include CIDs from earlier projects (shared CSV cache). For Moodify-only
# compound-level rates, restrict to CIDs that appear in the ingredient table.
_m = molecules_df["CID"].dropna()
moodify_cid_set = set(int(float(x)) for x in _m.unique())
props_moodify = props_df.loc[props_df["cid"].isin(moodify_cid_set)].copy()

err_merged = molecules_with_props["bp_c_pubchem_fetch_error"].fillna("")
err_props_m = props_moodify["error"].fillna("")

n_moodify_cids = len(moodify_cid_set)
n_props_rows_for_moodify = int(props_moodify["cid"].nunique())

summary = {
    # --- Ingredient-level (merged rows) ---
    "n_ingredient_rows": len(molecules_with_props),
    "rows_with_missing_cid": int(molecules_with_props["CID"].isna().sum()),
    "rows_bp_present_pct": round(molecules_with_props["bp_c_pubchem"].notna().mean() * 100, 2),
    "rows_vp_present_pct": round(molecules_with_props["vp_mmhg_pubchem"].notna().mean() * 100, 2),
    "rows_both_bp_and_vp_pct": round(
        (molecules_with_props["bp_c_pubchem"].notna() & molecules_with_props["vp_mmhg_pubchem"].notna()).mean()
        * 100,
        2,
    ),
    "rows_fetch_error_pct": round((err_merged != "").mean() * 100, 2),
    # --- Moodify compound-level (unique CIDs in this Excel, subset of props_df) ---
    "moodify_distinct_cids": n_moodify_cids,
    "moodify_cids_with_a_props_row": n_props_rows_for_moodify,
    "moodify_cids_still_missing_from_props_after_fetch": n_moodify_cids - n_props_rows_for_moodify,
    "moodify_cids_bp_present_pct": round(props_moodify["boiling_point_c"].notna().mean() * 100, 2)
    if n_props_rows_for_moodify
    else None,
    "moodify_cids_vp_present_pct": round(props_moodify["vapor_pressure_mmhg"].notna().mean() * 100, 2)
    if n_props_rows_for_moodify
    else None,
    "moodify_cids_both_bp_and_vp_pct": round(
        (props_moodify["boiling_point_c"].notna() & props_moodify["vapor_pressure_mmhg"].notna()).mean() * 100,
        2,
    )
    if n_props_rows_for_moodify
    else None,
    "moodify_cids_fetch_error_pct": round((err_props_m != "").mean() * 100, 2)
    if n_props_rows_for_moodify
    else None,
}

import json

print(json.dumps(summary, indent=2))

{
  "n_ingredient_rows": 2326,
  "rows_with_missing_cid": 0,
  "rows_bp_present_pct": 52.45,
  "rows_vp_present_pct": 25.75,
  "rows_both_bp_and_vp_pct": 23.34,
  "rows_fetch_error_pct": 0.0,
  "moodify_distinct_cids": 2006,
  "moodify_cids_with_a_props_row": 2006,
  "moodify_cids_still_missing_from_props_after_fetch": 0,
  "moodify_cids_bp_present_pct": 50.9,
  "moodify_cids_vp_present_pct": 23.28,
  "moodify_cids_both_bp_and_vp_pct": 21.19,
  "moodify_cids_fetch_error_pct": 0.0
}
